# 01 — Monte Carlo Methods
**Week 5 | Model-Free Learning**

Monte Carlo methods learn from **complete episodes** of experience — no model required.

The key insight: the value of a state is the **average return** we observed when visiting it.

$$V(s) \approx \text{average}\{G_t : s_t = s\}$$

**First-visit MC**: only count the first time we visit state s in each episode.

In [ ]:
try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys; subprocess.check_call([sys.executable,'-m','pip','install','gymnasium','-q']); import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. First-Visit MC Prediction

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=True)
n_states  = env.observation_space.n
n_actions = env.action_space.n

def generate_episode(env, policy, max_steps=200):
    """Run one episode and return (states, actions, rewards)."""
    s, _ = env.reset()
    states, actions, rewards = [], [], []
    for _ in range(max_steps):
        a = policy(s)
        ns, r, term, trunc, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r)
        s = ns
        if term or trunc: break
    return states, actions, rewards

def compute_returns(rewards, gamma=0.99):
    G, running = [], 0.0
    for r in reversed(rewards):
        running = r + gamma * running
        G.insert(0, running)
    return G

def first_visit_mc_prediction(env, policy, n_episodes=50_000, gamma=0.99):
    V = np.zeros(n_states)
    returns = {s: [] for s in range(n_states)}
    win_history = []
    for ep in range(n_episodes):
        states, _, rewards = generate_episode(env, policy)
        G_list = compute_returns(rewards, gamma)
        wins = int(rewards[-1] == 1.0)
        win_history.append(wins)
        visited = set()
        for s, G in zip(states, G_list):
            if s not in visited:
                visited.add(s)
                returns[s].append(G)
                V[s] = np.mean(returns[s])
    return V, win_history

random_policy = lambda s: env.action_space.sample()
print("Running first-visit MC prediction (random policy)...")
V_mc, wins = first_visit_mc_prediction(env, random_policy, 30_000)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(V_mc.reshape(4,4), cmap='RdYlGn')
plt.colorbar(im, ax=axes[0])
for s in range(16):
    r,c=divmod(s,4); axes[0].text(c,r,f'{V_mc[s]:.3f}',ha='center',va='center',fontsize=9)
axes[0].set_title('V(s) under random policy (MC)'); axes[0].set_xticks([]); axes[0].set_yticks([])

window = 500
rolling = np.convolve(wins, np.ones(window)/window, mode='valid')
axes[1].plot(rolling, color='steelblue', linewidth=1.5)
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Win rate (rolling 500-ep)')
axes[1].set_title('Random Policy — Win Rate over Time')
plt.tight_layout(); plt.show()

## 2. MC Control — ε-greedy Improvement

In [ ]:
def mc_control(env, n_episodes=80_000, gamma=0.99, eps_start=1.0, eps_end=0.01):
    Q = np.zeros((n_states, n_actions))
    returns_sum = np.zeros((n_states, n_actions))
    returns_cnt = np.zeros((n_states, n_actions))
    win_history = []

    for ep in range(n_episodes):
        eps = max(eps_end, eps_start - ep * (eps_start - eps_end) / n_episodes)
        def eps_greedy(s): return np.random.randint(n_actions) if np.random.rand()<eps else np.argmax(Q[s])

        states, actions, rewards = generate_episode(env, eps_greedy)
        G_list = compute_returns(rewards, gamma)
        win_history.append(int(rewards[-1]==1.0))

        visited = set()
        for s, a, G in zip(states, actions, G_list):
            if (s,a) not in visited:
                visited.add((s,a))
                returns_sum[s,a] += G
                returns_cnt[s,a] += 1
                Q[s,a] = returns_sum[s,a] / returns_cnt[s,a]
    return Q, win_history

print("Running MC control...")
Q_mc, wins_mc = mc_control(env)

In [ ]:
window = 1000
rolling_mc = np.convolve(wins_mc, np.ones(window)/window, mode='valid')
plt.figure(figsize=(8,3))
plt.plot(rolling_mc, color='steelblue', linewidth=1.5)
plt.xlabel('Episode'); plt.ylabel(f'Win rate (rolling {window}-ep window)')
plt.title('MC Control Learning Curve on Slippery FrozenLake')
plt.tight_layout(); plt.show()

# Evaluate final greedy policy
final_policy = Q_mc.argmax(axis=1)
wins_final = sum(1 for _ in range(1000) for s, _, _, _, _ in [env.step(final_policy[env.reset()[0]])])
print(f"Final policy win rate (greedy): {np.mean([generate_episode(env, lambda s: Q_mc.argmax(axis=1)[s])[-1][-1]==1.0 for _ in range(1000)]):.2%}")

## ✅ Exercises
1. Compare first-visit vs every-visit MC. Do they converge to the same V?
2. Plot how Q-values for state 0 change over episodes. When do they stabilise?
3. **Challenge**: implement **off-policy MC** with importance sampling. Evaluate a target policy (greedy) while following a behaviour policy (random).

## Q1

In [ ]:
def every_visit_mc_prediction(env, policy, n_episodes=30_000, gamma=0.99):
    V = np.zeros(n_states)
    returns = {s: [] for s in range(n_states)}
    for ep in range(n_episodes):
        states, _, rewards = generate_episode(env, policy)
        G_list = compute_returns(rewards, gamma)
        for s, G in zip(states, G_list):   # no visited check — every visit counted
            returns[s].append(G)
            V[s] = np.mean(returns[s])
    return V

V_every = every_visit_mc_prediction(env, random_policy)

print("Max difference between First-Visit and Every-Visit V:")
print(np.max(np.abs(V_mc - V_every)))

Yes, both converge to the same V(s). First-visit and every-visit MC are both unbiased estimators of the true value function — they differ only in which samples they use per episode. With enough episodes, both averages converge to the true expected return under the policy.

## Q2 

Q-values for state 0 typically stabilise around 30,000–50,000 episodes. Early on they fluctuate heavily due to sparse wins (reward=1 only at goal) and high exploration. As ε decays and more returns are averaged, the estimates settle.

In [ ]:
def mc_control_track_q0(env, n_episodes=80_000, gamma=0.99, eps_start=1.0, eps_end=0.01):
    Q = np.zeros((n_states, n_actions))
    returns_sum = np.zeros((n_states, n_actions))
    returns_cnt = np.zeros((n_states, n_actions))
    q0_history  = []

    for ep in range(n_episodes):
        eps = max(eps_end, eps_start - ep * (eps_start - eps_end) / n_episodes)
        def eps_greedy(s): return np.random.randint(n_actions) if np.random.rand() < eps else np.argmax(Q[s])

        states, actions, rewards = generate_episode(env, eps_greedy)
        G_list = compute_returns(rewards, gamma)

        visited = set()
        for s, a, G in zip(states, actions, G_list):
            if (s, a) not in visited:
                visited.add((s, a))
                returns_sum[s, a] += G
                returns_cnt[s, a] += 1
                Q[s, a] = returns_sum[s, a] / returns_cnt[s, a]

        q0_history.append(Q[0].copy())
    return Q, np.array(q0_history)

Q_tracked, q0_hist = mc_control_track_q0(env)

plt.figure(figsize=(9, 3.5))
action_labels = ['Left', 'Down', 'Right', 'Up']
for a in range(n_actions):
    plt.plot(q0_hist[:, a], linewidth=1, label=action_labels[a])
plt.xlabel('Episode'); plt.ylabel('Q(s=0, a)')
plt.title('Q-values for State 0 over Episodes')
plt.legend(fontsize=9); plt.tight_layout(); plt.show()

## Q3 

In [ ]:
def off_policy_mc(env, n_episodes=100_000, gamma=0.99):
    """
    Behaviour policy: random (b)
    Target policy:    greedy w.r.t Q (pi)
    Uses weighted importance sampling to evaluate target policy.
    """
    Q = np.zeros((n_states, n_actions))
    C = np.zeros((n_states, n_actions))  # cumulative importance weights

    for ep in range(n_episodes):
        # Behaviour policy: uniform random
        b_policy = lambda s: np.random.randint(n_actions)
        states, actions, rewards = generate_episode(env, b_policy)
        G = 0.0
        W = 1.0  # importance sampling ratio

        for t in reversed(range(len(states))):
            s, a, r = states[t], actions[t], rewards[t]
            G = r + gamma * G

            C[s, a] += W
            Q[s, a] += (W / C[s, a]) * (G - Q[s, a])

            # Target policy is greedy — if action doesn't match greedy, ratio = 0, stop
            if a != np.argmax(Q[s]):
                break

            # b(a|s) = 1/n_actions for uniform random behaviour policy
            W *= n_actions   # W = W * (pi(a|s) / b(a|s)) = W * (1 / (1/n_actions))

    return Q

Q_offpolicy = off_policy_mc(env)
final_policy_op = Q_offpolicy.argmax(axis=1)

win_rate = np.mean([
    generate_episode(env, lambda s: final_policy_op[s])[-1][-1] == 1.0
    for _ in range(1000)
])
print(f"Off-policy MC win rate: {win_rate:.2%}")

The importance sampling ratio W corrects for the mismatch between the behaviour policy (random) and the target policy (greedy). The episode is truncated early whenever the taken action does not match the target policy's greedy action, since the ratio becomes zero beyond that point — this is standard in ordinary and weighted importance sampling for off-policy MC.